In [3]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

In [2]:
!pip install scikit-learn pandas

  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl (8.1 MB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl (36.6 MB)
  Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)



[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
df = pd.read_csv('../laptop.csv')

In [5]:
df = df.dropna(subset=['price'])


In [ ]:
# 2. Préparer les données pour ML
prices = df[['price']]

In [8]:

# 3. Clustering K-Means (Segments de prix)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(prices)

In [9]:
# Assigner des labels lisibles (Bas, Milieu, Haut)
cluster_means = df.groupby('cluster')['price'].mean().sort_values()
label_map = {cluster_means.index[0]: 'Entrée de gamme', 
             cluster_means.index[1]: 'Milieu de gamme', 
             cluster_means.index[2]: 'Haut de gamme'}
df['cluster_label'] = df['cluster'].map(label_map)

In [10]:
# 4. Détection d'anomalies (Offres suspectes / Fausses annonces)
iso = IsolationForest(contamination=0.05, random_state=42)
df['is_anomaly'] = iso.fit_predict(prices) == -1

In [11]:
# 5. Afficher les résultats
print("Anomalies détectées :", df['is_anomaly'].sum())
print(df[['title', 'price', 'cluster_label', 'is_anomaly']].head(10))

Anomalies détectées : 7
                                               title   price    cluster_label  \
0  Dell Latitude 14" Laptop Computer Intel i7 Up ...  334.03  Entrée de gamme   
1  Dell Latitude 14" Laptop Computer Intel i5 Up ...  210.98  Entrée de gamme   
2  Dell Latitude 15.6” FHD Core i5 Laptop PC Up T...  250.73  Entrée de gamme   
3  Dell Inspiron 14 3420 14" Snapdragon 8cx Gen 2...  169.99  Entrée de gamme   
4  Auusda 14.1'' New Laptop, Intel N5095, 8GB RAM...  299.99  Entrée de gamme   
5  Dell Latitude 7490 Quad Core i5 16GB 512GB SSD...  219.99  Entrée de gamme   
6  Wholesale Lot of 10 Evolve III Maestro e-Book ...  259.00  Entrée de gamme   
7  Dell Latitude 7490 Intel Core i5 256G Windows ...  224.99  Entrée de gamme   
8  14" Dell Latitude Laptop i5 Dual Core Up To 16...  163.55  Entrée de gamme   
9  Dell Latitude 3445 Chromebook 8GB RAM 256GB SS...   94.99  Entrée de gamme   

   is_anomaly  
0       False  
1       False  
2       False  
3       False  
4   